In [ ]:
from pathlib import Path


import re
from pathlib import Path
from collections import Counter

import pandas as pd
import simplemma

# =============================================================================
# CONFIG
# =============================================================================

CONFIG = {
    "input_csv": "./data/transformed/digitalhaushalt_transformed_with_titel_text.csv",
    "output_dir": "./data/transformed/keyword_outputs",

    "text_col": "titel_text",
    "output_keyword_col": "keywords",

    # 0 = max_depth / innerste Ebene je einzelnem {}-Block
    # 1 = max_depth - 1
    # 2 = max_depth - 2
    # 3 = max_depth - 3
    "depth_offsets": [1],

    # Extraction
    "extract_braces": True,
    "extract_brackets": True,

    # Postprocessing
    "normalize_whitespace": True,
    "normalize_adjective_first_word": True,
    "fix_case_single_words": True,
    "strip_trailing_stopwords": True,
    "filter_noise_words": True,
    "exact_dedup_per_row": True,

    # Optional. Default bewusst False.
    "levenshtein_merge_d1": False,

    # Global normalization after row-level extraction.
    # Frequency-based, based on the actually extracted vocabulary.
    "global_normalize_case": True,
    "global_normalize_hyphen_space": True,
    "global_normalize_single_word_endings": True,
    "print_global_merges": True,

    "encoding": "utf-8-sig",
}


TRAILING_STOPWORDS = {
    "für", "und", "oder", "der", "die", "das", "des", "dem", "den",
    "von", "mit", "an", "in", "zu", "im", "am", "bei", "nach",
    "aus", "auf", "vor", "seit", "über", "unter",
}

GLUED_TRAILING_STOPWORDS = {
    "und", "oder", "für", "zur", "zum", "von", "mit", "nach"
}

NOISE_SUBSTRINGS = {
    "haushaltsvermerk",
    "verpflichtungsermächtigung",
    "haushaltsansatz",
    "haushaltsunterlage",
    "erläuterung",
    "erläuterungen",
}

WORD_RE = re.compile(r"[A-Za-zÄÖÜäöüß]{2,}")


# =============================================================================
# BASIC HELPERS
# =============================================================================

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()


def strip_outer_parentheses(s: str) -> str:
    return re.sub(r"^\(|\)$", "", str(s)).strip()


def has_word(s: str) -> bool:
    return bool(WORD_RE.search(str(s)))


def remove_all_markup_chars(s: str) -> str:
    return str(s).replace("{", "").replace("}", "").replace("[", "").replace("]", "")


def clean_leftover_markup(term: str) -> str:
    return normalize_space(remove_all_markup_chars(term))


def remove_leading_dash(term: str) -> str:
    return re.sub(r"^\s*[-–—]\s*", "", term).strip()

def strip_trailing_dash(term: str) -> str:
    return re.sub(r"\s*[-–—]\s*$", "", term).strip()


def strip_edge_punctuation(term: str) -> str:
    return str(term).strip().strip('"\'“”„,;:.')

def greedy_trailing(text: str, pos: int) -> str:
    """
    Sammelt direkt anschließende Wortzeichen nach } oder ].

    Beispiele:
    {Plattform}en -> trailing 'en'
    {{{Digital}e} Technologien} -> trailing ''
    """
    trailing = ""
    k = pos

    while k < len(text) and re.match(r"[A-Za-zÄÖÜäöüß0-9-]", text[k]):
        trailing += text[k]
        k += 1

    return trailing


def strip_glued_trailing_stopword(term: str) -> str:
    """
    Generisch:
    ITund -> IT
    Softwarefür -> Software
    Plattformenund -> Plattformen

    Keine fachlichen Spezialfälle.
    """
    term = term.strip()
    lower = term.lower()

    for sw in sorted(GLUED_TRAILING_STOPWORDS, key=len, reverse=True):
        if lower.endswith(sw) and len(term) > len(sw) + 1:
            base = term[:-len(sw)].strip()
            if has_word(base):
                return base

    return term


def looks_like_broken_fragment(term: str) -> bool:
    """
    Generische Fragment-Erkennung.
    Keine fachlichen Keyword-Hacks.
    """
    t = normalize_space(term)

    if not t:
        return True

    # Nach Cleaning darf kein Markup übrig sein.
    if any(c in t for c in "{}[]"):
        return True

    # Neu[e Mobilität] darf nicht als "e Mobilität" extrahiert werden.
    if re.match(r"^[a-zäöü]{1,2}\s+[A-ZÄÖÜ]", t):
        return True

    # Reine technische Präfix-Fragmente.
    if t.lower() in {"it-", "ki-", "ikt-", "e", "i"}:
        return True

    return False

# =============================================================================
# DEPTH LOGIC
# =============================================================================

def get_max_depth(brace_str: str) -> int:
    depth = 0
    max_seen = 0

    for char in brace_str:
        if char == "{":
            depth += 1
            max_seen = max(max_seen, depth)
        elif char == "}":
            depth -= 1

    return max_seen


def first_child_block(brace_str: str) -> str | None:
    """
    Gibt den ersten direkten inneren {}-Block zurück.

    Beispiel:
    {{{Digital}e} Technologien}
    -> {{Digital}e}
    """
    s = brace_str[1:-1] if brace_str.startswith("{") and brace_str.endswith("}") else brace_str

    i = 0
    while i < len(s):
        if s[i] != "{":
            i += 1
            continue

        depth = 0
        j = i

        while j < len(s):
            if s[j] == "{":
                depth += 1
            elif s[j] == "}":
                depth -= 1
                if depth == 0:
                    return s[i:j + 1]
            j += 1

        return None

    return None


def remove_braces(s: str) -> str:
    return re.sub(r"[{}]", "", s).strip()


def term_at_level(brace_str: str, level: int) -> str:
    """
    level=1         -> äußerste Ebene / spezifischste Form
    level=max_depth -> innerste Ebene / allgemeinste Form

    Beispiel:
    {{{Digital}e} Technologien}

    level 1 -> Digitale Technologien
    level 2 -> Digitale
    level 3 -> Digital
    """
    if level <= 1:
        return remove_braces(brace_str)

    child = first_child_block(brace_str)
    if child is None:
        return remove_braces(brace_str)

    return term_at_level(child, level - 1)


def target_level_for_offset(brace_str: str, depth_offset: int) -> int:
    depth = get_max_depth(brace_str)
    return max(1, depth - depth_offset)


def term_for_depth_offset(brace_str: str, depth_offset: int) -> tuple[str, int]:
    target_level = target_level_for_offset(brace_str, depth_offset)
    return term_at_level(brace_str, target_level), target_level


# =============================================================================
# EXTRACTION
# =============================================================================

def extract_brace_blocks(text: str) -> list[tuple[str, str]]:
    """
    Extrahiert vollständige äußere {}-Blöcke.

    Wichtig:
    Für das {}-Parsing werden [] entfernt.
    Grund: TexAn-Mischformen wie [-{{{{Digital}]e} Innovation}en}
    parsen sonst kaputt.
    """
    text = str(text).replace("[", "").replace("]", "")

    results = []
    i = 0
    n = len(text)

    while i < n:
        if text[i] != "{":
            i += 1
            continue

        depth = 0
        j = i

        while j < n:
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
                if depth == 0:
                    break
            j += 1

        if j >= n or depth != 0:
            i += 1
            continue

        brace_str = text[i:j + 1]
        trailing = greedy_trailing(text, j + 1)

        preview = clean_leftover_markup(remove_braces(brace_str))
        preview = remove_leading_dash(preview)

        if has_word(preview) and not looks_like_broken_fragment(preview):
            results.append((brace_str, trailing))

        i = j + 1

    return results


def extract_bracket_terms(text: str) -> list[str]:
    """
    Extrahiert echte []-Keywords.

    Wenn im []-Inhalt { oder } vorkommt, wird er ignoriert,
    weil es dann eine Misch-Markierung ist, die über {} verarbeitet wird.
    """
    text = str(text)
    results = []

    for match in re.finditer(r"\[([^\]]+)\]", text):
        content = match.group(1).strip()

        if "{" in content or "}" in content:
            continue

        content = content.strip("[]() ")
        content = remove_leading_dash(content)

        if looks_like_broken_fragment(content):
            continue

        trailing = greedy_trailing(text, match.end())
        term = strip_outer_parentheses(content + trailing)
        term = clean_leftover_markup(term)
        term = remove_leading_dash(term)
        term = strip_glued_trailing_stopword(term)

        if has_word(term) and not looks_like_broken_fragment(term):
            results.append(term)

    return results


def extract_row_terms(text: str, cfg: dict) -> dict:
    if not isinstance(text, str):
        text = ""

    return {
        "braces": extract_brace_blocks(text) if cfg["extract_braces"] else [],
        "brackets": extract_bracket_terms(text) if cfg["extract_brackets"] else [],
    }


# =============================================================================
# POSTPROCESSING
# =============================================================================

def normalize_adjective_first_word(term: str) -> str:
    """
    Einfache Normalisierung des ersten Wortes:
    Künstlicher Intelligenz -> Künstliche Intelligenz
    digitaler Technologien -> digitale Technologien

    Bewusst nur erstes Wort, keine aggressive Lemmatisierung.
    """
    words = term.split()

    if len(words) < 2:
        return term

    first = words[0]
    lower = first.lower()

    for ending in ["er", "em", "en", "es"]:
        if lower.endswith(ending) and len(lower) - len(ending) >= 3:
            stem = first[:-len(ending)]
            normalized = stem + "e"
            return " ".join([normalized] + words[1:])

    return term


def fix_case_single_word(term: str) -> str:
    """
    Einwort-Begriffe großschreiben, außer technische Schreibweisen.
    """
    if not term or " " in term:
        return term

    if re.match(r"^(go-|e-|i-|de:|dtec|open|chat|gpt|ki-|it-|ikt-)", term, re.I):
        return term

    if term[0].islower():
        return term[0].upper() + term[1:]

    return term


def strip_trailing_stopwords(term: str) -> str:
    words = term.split()

    while words and words[-1].lower() in TRAILING_STOPWORDS:
        words.pop()

    return " ".join(words)


def contains_noise_word(term: str) -> bool:
    lower = term.lower()
    return any(noise in lower for noise in NOISE_SUBSTRINGS)


def postprocess_term(term: str, cfg: dict) -> str:
    term = strip_outer_parentheses(term)
    term = clean_leftover_markup(term)
    term = strip_edge_punctuation(term)
    term = remove_leading_dash(term)
    term = strip_glued_trailing_stopword(term)

    if cfg["normalize_whitespace"]:
        term = normalize_space(term)

    if cfg["normalize_adjective_first_word"]:
        term = normalize_adjective_first_word(term)

    if cfg["fix_case_single_words"]:
        term = fix_case_single_word(term)

    if cfg["strip_trailing_stopwords"]:
        term = strip_trailing_stopwords(term)

    term = clean_leftover_markup(term)
    term = strip_edge_punctuation(term)
    term = remove_leading_dash(term)
    term = strip_glued_trailing_stopword(term)

    if cfg["normalize_whitespace"]:
        term = normalize_space(term)

    term = strip_trailing_dash(term)
    term = strip_edge_punctuation(term)

    if cfg["normalize_whitespace"]:
        term = normalize_space(term)

    if cfg["filter_noise_words"] and contains_noise_word(term):
        return ""

    if looks_like_broken_fragment(term):
        return ""

    if not has_word(term):
        return ""

    return term


def dedup_terms(terms: list[str]) -> list[str]:
    seen = {}

    for term in terms:
        key = term.lower()
        if key not in seen:
            seen[key] = term

    return list(seen.values())


# =============================================================================
# GLOBAL NORMALIZATION
# =============================================================================

def choose_preferred_form(forms: list[str], counts: Counter) -> str:
    """
    Winner-Regel:
    1. häufigste Form
    2. bei Gleichstand: bereinigte Form ohne Randzeichen/trailing dash
    3. bei Gleichstand: kürzere Form
    """
    unique_forms = list(dict.fromkeys(forms))

    # def score(term: str):
    #     clean = strip_trailing_dash(strip_edge_punctuation(term))
    #     return (
    #         counts[term],
    #         int(term == clean),
    #         -len(term),
    #     )

    def score(term: str):
        clean = strip_trailing_dash(strip_edge_punctuation(term))
        return (
            -len(term),
            int(not term.isupper()),   # prefer non-ALLCAPS at equal length
            counts[term],
            int(term == clean),
        )

    return max(unique_forms, key=score)


def key_case(term: str) -> str:
    return normalize_space(term).lower()


def key_hyphen_space(term: str) -> str:
    """
    Gruppiert Varianten wie:
    Open Source / Open-Source
    Smart City / Smart-City
    High Performance / High-Performance
    """
    key = term.lower()
    key = re.sub(r"\s*[-–—]\s*", " ", key)
    key = normalize_space(key)
    return key


# def key_single_word_ending(term: str) -> str | None:
#     """
#     Grober Stamm für Einwort-Varianten.
#     Wichtig: Hier wird NICHT direkt abgeschnitten.
#     Es wird nur gruppiert. Ersetzt wird später durch die häufigste echte Form.

#     Beispiele:
#     Digital / Digitale / Digitaler / Digitalen -> digital
#     Datenbank / Datenbanken -> datenbank
#     Digitalisierung / Digitalisierungs -> digitalisierung
#     """
#     t = strip_edge_punctuation(term)

#     if " " in t or "-" in t:
#         return None

#     if not re.fullmatch(r"[A-Za-zÄÖÜäöüß]{6,}", t):
#         return None

#     lower = t.lower()

#     for ending in ["ern", "ers", "er", "es", "em", "en", "e", "n", "s"]:
#         if lower.endswith(ending) and len(lower) - len(ending) >= 5:
#             return lower[:-len(ending)]

#     return lower

def key_single_word_ending(term: str) -> str | None:
    """
    Groups single-word inflection variants by their lemma.
    Digital/Digitale/Digitalen -> lemma 'digital'
    Drohne/Drohnen -> lemma 'drohne'
    Compute and Computer have different lemmas -> NOT merged.
    """
    t = strip_edge_punctuation(term)
    if " " in t or "-" in t:
        return None
    if not re.fullmatch(r"[A-Za-zÄÖÜäöüß]{4,}", t):
        return None
    return simplemma.lemmatize(t.lower(), lang="de")


def add_group_replacements(groups: dict[str, list[str]], counts: Counter, replacement: dict[str, str]) -> None:
    for _, forms in groups.items():
        unique_forms = list(dict.fromkeys(forms))

        if len(unique_forms) < 2:
            continue

        winner = choose_preferred_form(unique_forms, counts)

        for form in unique_forms:
            if form != winner:
                replacement[form] = winner


def build_global_normalization_map(all_keywords: list[str], cfg: dict) -> dict[str, str]:
    counts = Counter(all_keywords)
    replacement: dict[str, str] = {}

    if cfg.get("global_normalize_case", True):
        groups = {}
        for kw in counts:
            groups.setdefault(key_case(kw), []).append(kw)
        add_group_replacements(groups, counts, replacement)

    if cfg.get("global_normalize_hyphen_space", True):
        groups = {}
        for kw in counts:
            groups.setdefault(key_hyphen_space(kw), []).append(kw)
        add_group_replacements(groups, counts, replacement)

    if cfg.get("global_normalize_single_word_endings", True):
        groups = {}
        for kw in counts:
            key = key_single_word_ending(kw)
            if key:
                groups.setdefault(key, []).append(kw)
        add_group_replacements(groups, counts, replacement)

    return replacement


def apply_global_normalization(keyword_rows: list[list[str]], cfg: dict, label: str = "") -> list[list[str]]:
    flat = [kw for row in keyword_rows for kw in row if kw]
    replacement = build_global_normalization_map(flat, cfg)

    if cfg.get("print_global_merges", True) and replacement:
        print(f"  Global normalization merges{f' ({label})' if label else ''}: {len(replacement)}")
        for src, dst in sorted(replacement.items(), key=lambda x: (x[1].lower(), x[0].lower()))[:80]:
            print(f"    {src!r} -> {dst!r}")
        if len(replacement) > 80:
            print(f"    ... {len(replacement) - 80} more")

    normalized_rows = []
    for row in keyword_rows:
        normalized = [replacement.get(kw, kw) for kw in row]
        normalized_rows.append(dedup_terms(normalized))

    return normalized_rows


# =============================================================================
# OPTIONAL LEVENSHTEIN D=1
# =============================================================================

def levenshtein_distance_limited(a: str, b: str, max_dist: int = 1) -> int:
    if abs(len(a) - len(b)) > max_dist:
        return max_dist + 1

    a = a.lower()
    b = b.lower()

    if a == b:
        return 0

    prev = list(range(len(b) + 1))

    for i, ca in enumerate(a, start=1):
        curr = [i]

        for j, cb in enumerate(b, start=1):
            curr.append(
                min(
                    prev[j] + 1,
                    curr[j - 1] + 1,
                    prev[j - 1] + (ca != cb),
                )
            )

        if min(curr) > max_dist:
            return max_dist + 1

        prev = curr

    return prev[-1]


def build_levenshtein_d1_map(all_keywords: list[str]) -> dict[str, str]:
    counts = Counter(all_keywords)
    candidates = sorted(
        [k for k, c in counts.items() if c >= 2 and len(k) >= 8],
        key=str.lower,
    )

    simple_suffixes = ("e", "en", "er", "em", "es", "n", "s")
    replacement = {}

    for i, a in enumerate(candidates):
        for b in candidates[i + 1:]:
            if abs(len(a) - len(b)) > 1:
                continue

            al = a.lower()
            bl = b.lower()

            shorter, longer = (al, bl) if len(al) <= len(bl) else (bl, al)
            suffix = longer[len(shorter):]

            if suffix in simple_suffixes:
                continue

            if levenshtein_distance_limited(a, b, max_dist=1) == 1:
                winner, loser = (a, b) if counts[a] >= counts[b] else (b, a)
                replacement[loser.lower()] = winner

    return replacement


def apply_replacement_map(keyword_rows: list[list[str]], replacement: dict[str, str]) -> list[list[str]]:
    new_rows = []

    for terms in keyword_rows:
        replaced = [replacement.get(term.lower(), term) for term in terms]
        new_rows.append(dedup_terms(replaced))

    return new_rows


# =============================================================================
# DATASET BUILDING
# =============================================================================

def build_terms_for_offset(row_extracted: dict, depth_offset: int, cfg: dict) -> list[str]:
    terms = []

    for brace_str, trailing in row_extracted["braces"]:
        term, target_level = term_for_depth_offset(brace_str, depth_offset)

        # Trailing gehört nur zur äußersten/spezifischsten Form.
        # Wenn depth_offset so groß ist, dass target_level=1 wird, ergänzen.
        if target_level == 1 and trailing and not trailing.startswith("-") and not len(trailing) > 4:
            term = term + trailing

        term = postprocess_term(term, cfg)

        if term:
            terms.append(term)

    # []-Keywords haben keine Depth-Logik und werden in jeder Variante ergänzt.
    for bracket_term in row_extracted["brackets"]:
        term = postprocess_term(bracket_term, cfg)

        if term:
            terms.append(term)

    if cfg["exact_dedup_per_row"]:
        terms = dedup_terms(terms)

    return terms


def keyword_rows_to_strings(keyword_rows: list[list[str]]) -> list[str]:
    return ["|".join(row) for row in keyword_rows]


def build_frequency_df(keyword_rows: list[list[str]]) -> pd.DataFrame:
    flat = [kw for row in keyword_rows for kw in row if kw]
    counts = Counter(flat)

    return pd.DataFrame(
        [{"keyword": keyword, "count": count} for keyword, count in counts.items()]
    ).sort_values(["count", "keyword"], ascending=[False, True])


def build_alphabetical_df(keyword_rows: list[list[str]]) -> pd.DataFrame:
    flat = [kw for row in keyword_rows for kw in row if kw]
    counts = Counter(flat)

    return pd.DataFrame(
        [{"keyword": keyword, "count": count} for keyword, count in counts.items()]
    ).sort_values(["keyword"], ascending=True)


def build_comparison_df(all_outputs: dict[int, list[list[str]]]) -> pd.DataFrame:
    rows = []

    for offset, keyword_rows in all_outputs.items():
        freq = Counter([kw for row in keyword_rows for kw in row if kw])

        for keyword, count in freq.items():
            rows.append(
                {
                    "depth_offset": offset,
                    "dataset": f"level_{offset}",
                    "keyword": keyword,
                    "count": count,
                }
            )

    return pd.DataFrame(rows).sort_values(
        ["depth_offset", "count", "keyword"],
        ascending=[True, False, True],
    )


# =============================================================================
# MAIN
# =============================================================================

def run_pipeline(cfg: dict) -> None:
    input_csv = Path(cfg["input_csv"])
    output_dir = Path(cfg["output_dir"])
    # output_dir.mkdir(parents=True, exist_ok=True)   # entfernt: kein gitignored-Ordner

    print("Loading data...")
    df = pd.read_csv(input_csv, dtype=str)
    print(f"Rows: {len(df):,}")

    if cfg["text_col"] not in df.columns:
        raise ValueError(f"Column not found: {cfg['text_col']}")

    print("Extracting raw terms...")
    extracted_rows = [
        extract_row_terms(text, cfg)
        for text in df[cfg["text_col"]].fillna("")
    ]

    all_outputs: dict[int, list[list[str]]] = {}

    for offset in cfg["depth_offsets"]:
        print(f"Building level_{offset}...")

        keyword_rows = [
            build_terms_for_offset(row_extracted, offset, cfg)
            for row_extracted in extracted_rows
        ]

        if cfg["levenshtein_merge_d1"]:
            flat = [kw for row in keyword_rows for kw in row if kw]
            replacement = build_levenshtein_d1_map(flat)

            if replacement:
                print(f"  Levenshtein d=1 merges: {len(replacement)}")
                for loser, winner in sorted(replacement.items()):
                    print(f"    {loser!r} -> {winner!r}")

                keyword_rows = apply_replacement_map(keyword_rows, replacement)

        keyword_rows = apply_global_normalization(
            keyword_rows,
            cfg,
            label=f"level_{offset}",
        )

        all_outputs[offset] = keyword_rows

        df_out = df.copy()
        df_out[cfg["output_keyword_col"]] = keyword_rows_to_strings(keyword_rows)
        df_out["n_keywords"] = [len(row) for row in keyword_rows]

        out_csv = output_dir / f"keywords_level_{offset}.csv"
        run_pipeline.result = df_out   # Übergabe im Speicher statt Zwischendatei

        freq_df = build_frequency_df(keyword_rows)
        freq_csv = output_dir / f"keyword_frequencies_level_{offset}.csv"
        # freq_df.to_csv(freq_csv, index=False, encoding=cfg["encoding"])   # entfernt: nicht benötigte Nebendatei

        alpha_df = build_alphabetical_df(keyword_rows)
        alpha_csv = output_dir / f"keyword_alphabetical_level_{offset}.csv"
        # alpha_df.to_csv(alpha_csv, index=False, encoding=cfg["encoding"])   # entfernt: nicht benötigte Nebendatei

        # print(f"  Output: {out_csv}")
        print(f"  Freq:   {freq_csv}")
        print(f"  Unique keywords: {len(freq_df):,}")

    comparison_df = build_comparison_df(all_outputs)
    comparison_csv = output_dir / "keyword_frequencies_comparison_all_depths.csv"
    # comparison_df.to_csv(comparison_csv, index=False, encoding=cfg["encoding"])   # entfernt: nicht benötigte Nebendatei

    print()
    print("=" * 70)
    print("DONE")
    print("=" * 70)
    print(f"Comparison: {comparison_csv}")

    print()
    print("Top 20 per dataset:")
    for offset in cfg["depth_offsets"]:
        freq_df = build_frequency_df(all_outputs[offset])
        print()
        print(f"level_{offset}")
        print(freq_df.head(20).to_string(index=False))


if __name__ == "__main__":
    run_pipeline(CONFIG)


In [ ]:
import pandas as pd
from collections import Counter

# =============================================================================
# FINAL CLEANING — works on any level output (set LEVEL)
# =============================================================================

LEVEL = 1
OUTPUT_CSV = "./data/transformed/digitalhaushalt_transformed_with_titel_text_and_extracted_keywords_cleaned.csv"
KEYWORD_COL = "keywords"

SAFE_INFLECTION_ENDINGS = ["ungen", "en", "s"]
MIN_STEM_LENGTH = 6

PROTECTED_ENDINGS = (
    "daten",
    "basis",
    "fonds",
    "konsens",
    "eucaris",
)

# Final manual correction layer.
# Only for obvious cases where automatic lemmatizers produced too many errors.
MANUAL_MERGE_MAP = {
    "Abbiegeassistenzsystemen": "Abbiegeassistenzsysteme",
    "Algorithmen": "Algorithmus",
    "Bildgeneratoren": "Bildgenerator",
    "Bildungskompetenzzentren": "Bildungskompetenzzentrum",
    "Büromaschinen": "Büromaschine",
    "Computerspielen": "Computerspiel",
    "Cyberangriffen": "Cyberangriffe",
    "Datenerhebungen": "Datenerhebung",
    "Datenkompetenzzentren": "Datenkompetenzzentrum",
    "Datenleitungen": "Datenleitung",
    "Datenmengen": "Datenmenge",
    "Datenquellen": "Datenquelle",
    "Datenräumen": "Datenraum",
    "Datenverarbeitungsanlagen": "Datenverarbeitungsanlage",
    "Digitalgestützten": "Digitalgestützte",
    "Digitalisierten": "Digitalisierte",
    "Disruptiven": "Disruptive",
    "Echtzeitanalysatoren": "Echtzeitanalysator",
    "Elektroniksystemen": "Elektroniksystem",
    "Fernmeldeagenturen": "Fernmeldeagentur",
    "Fernmeldeanlagen": "Fernmeldeanlage",
    "Fernmeldeeinrichtungen": "Fernmeldeeinrichtung",
    "Fernmeldeleitungen": "Fernmeldeleitung",
    "Funkeinrichtungen": "Funkeinrichtung",
    "Informationsinfrastrukturen": "Informationsinfrastruktur",
    "Informationstechnologien": "Informationstechnologie",
    "Internetbasierten": "Internetbasierte",
    "KIMethoden": "KI-Methoden",
    "Kommunikationstechnologien": "Kommunikationstechnologie",
    "Netzkomponenten": "Netzkomponente",
    "Netztechnologien": "Netztechnologie",
    "Netzwerkkomponenten": "Netzwerkkomponente",
    "Quantentechnologien": "Quantentechnologie",
    "Reallaborbedingungen": "Reallaborbedingung",
    "Sensoren": "Sensor",
    "Skalierbaren": "Skalierbare",
    "Spezialprozessoren": "Spezialprozessor",
    "Standleitungen": "Standleitung",
    "Technologieplattformen": "Technologieplattform",
    "Telekommunikativen": "Telekommunikative",
    "Telemedizinischen": "Telemedizinisch",
    "Verschlüsselten": "Verschlüsselte",
    "Videokonferenzanlagen": "Videokonferenzanlage",
    "Weltraumgestützten": "Weltraumgestützte",
}

# Truncated / glued artifacts from the TexAn source (cannot be fixed by rule).
TRUNCATION_FIXES = {
    "ITan": "IT", "ITdes": "IT", "IThat": "IT", "ITist": "IT",
    "KIin": "KI", "KIbei": "KI", "KIvor": "KI",
    "DLRdas": "DLR", "IKTist": "IKT", "Fernmeldebzw": "Fernmelde",
    "generative KIvor": "generative KI",
    "Blockcha": "Blockchain",
    "Datenank": "Datenankauf",
    "Datenprovi": "Datenprovider",
    "Digital Tw": "Digital Twin",
    "Individualisierte Mediz": "Individualisierte Medizin",
    "Netze des Bun": "Netze des Bundes",
    "Telemediz": "Telemedizin",
    "KI-Metho": "KI-Methoden",
    "KIMetho": "KI-Methoden",
    "Mobilfunkausb": "Mobilfunkausbau",
    "Elektr. Schnittstelle Behör": "Elektr. Schnittstelle Behörden",
    "Cente for Intelligence and Security Studies": "Center for Intelligence and Security Studies",
    "Cybe Innovation Hub": "Cyber Innovation Hub",
    "Gree Ai": "Green Ai",
    "Gree IT": "Green IT",
    "HERKULe Folgeprojekt": "HERKULES Folgeprojekt",
    "HERKULe Folgeprojektes": "HERKULES Folgeprojekt",
    "Miteinande durch Innovation": "Miteinander durch Innovation",
    "Wissensspeiche Forschungsdaten": "Wissensspeicher Forschungsdaten",
    "Sondervermöge Infrastruktur": "Sondervermögen Infrastruktur",
    "Portalverb": "Portalverbund",
    "Rechenzentrumsverb": "Rechenzentrumsverbund",
    "Sovereign Tech F": "Sovereign Tech Fund",
    "Nutzerkonto B": "Nutzerkonto Bund",
    "Digitalfun": "Digitalfunk",
    "Hochleistu": "Hochleistung",
    "Date intelligent geteilt": "Daten intelligent geteilt",
    "Date zum Nutzen": "Daten zum Nutzen",
    "Ferdinand-Braun-Institu": "Ferdinand-Braun-Institut",
    "Serversorgung": "Serverversorgung",
    "AppDie": "App",
}

MANUAL_MERGE_MAP.update(TRUNCATION_FIXES)

SIMILARITY_FIXES = {
    "Platform": "Plattform",
    "Information System": "Informationssystem",
    "BigData": "Big Data",
    "Information Modelling": "Information Modeling",
    "Datensoveränität": "Datensouveränität",
    "Polizei 20/20": "Polizei 2020",
    "OpenRAN": "Open RAN",
    "Digitalgipfel": "Digital-Gipfel",

    "Satellite": "Satellit",
    "Start-ups": "Start-up",
    "Startup": "Start-up",
    "Kommunikationssysteme": "Kommunikationssystem",
    "Datenökosysteme": "Datenökosystem",
    "Elektroniksysteme": "Elektroniksystem",
    "IT-Systems": "IT-System",
    "IT-Projekte": "IT-Projekt",
    "KI-Mitteln": "KI-Mittel",
    "IT-Planungsrats": "IT-Planungsrat",
    "Weizenbaum-Instituts": "Weizenbaum-Institut",
    "IT-Dienstleistern": "IT-Dienstleister",
    "Datenlabore": "Datenlabor",
    "Reallabore": "Reallabor",

    "Deutsche Zentrums für Luft- und Raumfahrt": "Deutsche Zentrum für Luft- und Raumfahrt",
    "Deutsche Zentrums Mobilität der Zukunft": "Deutsche Zentrum Mobilität der Zukunft",
    "Digitalisierung Landbasierter Operationen": "Digitalisierung landbasierte Operationen",

    "Disruptiver": "Disruptive",
    "Skalierbarer": "Skalierbare",
    "Mikroelektronischer": "Mikroelektronische",
    "Serve": "Server",

    "Informationstechn": "Informationstechnik",
    "Datenräume": "Datenraum",
    "Robot": "Robotik",
    "Digitalpolit": "Digitalpolitik",
    "Kommunikationstechn": "Kommunikationstechnik",
    "Höchstleistungsrechnen": "Höchstleistungsrechn",
    "Hochleistungsrech": "Höchstleistungsrechn",
    "IT-Dienstleistungen": "IT-Dienstleistung",
    "best practices": "Best-Practice",
    'maschinelle Lerne': 'Maschinelle Lernen',
    'militärische Abschirmdienstes': 'Militärische Abschirmdienst'
}

MANUAL_MERGE_MAP.update(SIMILARITY_FIXES)


def split_keywords(cell: str) -> list[str]:
    if not isinstance(cell, str) or not cell.strip():
        return []
    return [k.strip() for k in cell.split("|") if k.strip()]


def strip_inflection(word: str) -> str | None:
    low = word.lower()

    if low.endswith(PROTECTED_ENDINGS):
        return None

    for ending in SAFE_INFLECTION_ENDINGS:
        if low.endswith(ending) and len(low) - len(ending) >= MIN_STEM_LENGTH:
            return word[:-len(ending)]

    return None


def is_comma_artifact(term: str) -> bool:
    return "," in term


def build_compound_merge_map(vocab: set, counts: Counter) -> dict:
    merge = {}

    for word in vocab:
        if " " in word or "-" in word:
            continue

        base = strip_inflection(word)

        if base and base in vocab and base != word:
            loser, winner = (word, base) if len(word) > len(base) else (base, word)
            merge[loser] = winner

    return merge


def build_safe_genitive_map(vocab: set, counts: Counter) -> dict:
    merge = {}

    for word in vocab:
        if " " in word or "-" in word:
            continue

        low = word.lower()

        if len(word) < 10:
            continue

        if word.isupper():
            continue

        if low.endswith(PROTECTED_ENDINGS):
            continue

        base = None

        if low.endswith("es"):
            base = word[:-2]
        elif low.endswith("s"):
            base = word[:-1]

        if base and len(base) >= 6 and base != word:
            merge[word] = base

    return merge


def build_manual_merge_map(vocab: set, compound_map: dict, genitive_map: dict) -> dict:
    return {
        loser: winner
        for loser, winner in MANUAL_MERGE_MAP.items()
        if loser in vocab
        and loser not in compound_map
        and loser not in genitive_map
    }


def dedup_case_insensitive(terms: list[str]) -> list[str]:
    seen = {}

    for term in terms:
        key = term.lower()
        if key not in seen:
            seen[key] = term

    return list(seen.values())


def run_cleaning(input_df, output_csv):
    df = input_df.copy()
    rows = [split_keywords(c) for c in df[KEYWORD_COL].fillna("")]

    flat = [k for row in rows for k in row]
    counts = Counter(flat)
    vocab = set(counts)

    print(f"Input: {len(df):,} rows, {len(vocab):,} unique keywords\n")

    comma_terms = sorted([k for k in vocab if is_comma_artifact(k)])
    compound_map = build_compound_merge_map(vocab, counts)
    genitive_map = build_safe_genitive_map(vocab, counts)

    genitive_map = {
        loser: winner
        for loser, winner in genitive_map.items()
        if loser not in compound_map
    }

    manual_map = build_manual_merge_map(vocab, compound_map, genitive_map)

    print(f"PROBLEM 1 — comma artifacts: {len(comma_terms)}")
    for k in comma_terms[:20]:
        print(f"    removing {k!r}")
    print()

    print(f"PROBLEM 2 — inflection variants with partner: {len(compound_map)}")
    for loser, winner in sorted(compound_map.items()):
        print(f"    {loser!r} ({counts[loser]}) -> {winner!r} ({counts[winner]})")
    print()

    print(f"PROBLEM 3 — safe genitive cleanup: {len(genitive_map)}")
    for loser, winner in sorted(genitive_map.items()):
        print(f"    {loser!r} ({counts[loser]}) -> {winner!r}")
    print()

    print(f"PROBLEM 4 — manual cleanup: {len(manual_map)}")
    for loser, winner in sorted(manual_map.items()):
        print(f"    {loser!r} ({counts[loser]}) -> {winner!r}")
    print()

    change_log = []
    cleaned_rows = []

    for row in rows:
        new_row = []

        for k in row:
            if is_comma_artifact(k):
                change_log.append((k, "<removed: comma>"))
                continue

            if k in compound_map:
                change_log.append((k, compound_map[k]))
                k = compound_map[k]

            if k in genitive_map:
                change_log.append((k, genitive_map[k]))
                k = genitive_map[k]

            if k in manual_map:
                change_log.append((k, manual_map[k]))
                k = manual_map[k]

            new_row.append(k)

        cleaned_rows.append(dedup_case_insensitive(new_row))

    df_out = df.copy()
    df_out[KEYWORD_COL] = ["|".join(r) for r in cleaned_rows]
    df_out.to_csv(output_csv, index=False, encoding="utf-8-sig")

    change_counts = Counter(change_log)

    change_df = pd.DataFrame(
        [{"from": a, "to": b, "count": c} for (a, b), c in change_counts.items()]
    ).sort_values("count", ascending=False)

    changes_csv = output_csv.replace(".csv", "_changes.csv")
    # change_df.to_csv(changes_csv, index=False, encoding="utf-8-sig")   # entfernt: nicht benötigte Nebendatei

    new_flat = [k for row in cleaned_rows for k in row]
    new_counts = Counter(new_flat)
    new_vocab = set(new_flat)

    freq_df = pd.DataFrame(
        [{"keyword": k, "count": c} for k, c in new_counts.items()]
    ).sort_values(["count", "keyword"], ascending=[False, True])

    freq_csv = output_csv.replace(".csv", "_frequencies.csv")
    # freq_df.to_csv(freq_csv, index=False, encoding="utf-8-sig")   # entfernt: nicht benötigte Nebendatei

    alpha_df = pd.DataFrame(
        [{"keyword": k, "count": c} for k, c in new_counts.items()]
    ).sort_values(["keyword"], ascending=[True])

    alpha_csv = output_csv.replace(".csv", "_alpha.csv")
    # alpha_df.to_csv(alpha_csv, index=False, encoding="utf-8-sig")   # entfernt: nicht benötigte Nebendatei

    print(f"Applied {len(change_log):,} changes")
    print(f"Output: {len(new_vocab):,} unique keywords (was {len(vocab):,})")
    print(f"Cleaned output: {output_csv}")
    print(f"Change log:      {changes_csv}")
    print(f"Frequencies:     {freq_csv}")
    print(f"Alphabetical:     {alpha_csv}")
    print()

    remaining = []

    for k in sorted(new_vocab):
        if " " in k or "-" in k:
            continue

        base = strip_inflection(k)

        if base and base not in new_vocab:
            remaining.append(k)

    print(f"REMAINING — inflected single words without partner/manual cleanup (left untouched): {len(remaining)}")
    for k in remaining:
        print(f"    {k!r}")

    print()
    print("Top 40 cleaned keywords:")
    print(freq_df.head(40).to_string(index=False))

    # ---- SIMILARITY REPORT (output only, changes nothing) ----
    def lev1(a, b):
        if abs(len(a) - len(b)) > 2:
            return 99
        a, b = a.lower(), b.lower()
        if a == b:
            return 0
        prev = list(range(len(b) + 1))
        for ca in a:
            cur = [prev[0] + 1]
            for j, cb in enumerate(b):
                cur.append(min(prev[j] + (ca != cb), cur[-1] + 1, prev[j + 1] + 1))
            prev = cur
        return prev[-1]

    cand = sorted([k for k in new_vocab if len(k) >= 5], key=str.lower)
    seen_pairs = set()
    pairs = []
    for i, a in enumerate(cand):
        for b in cand[i + 1:]:
            if abs(len(a) - len(b)) > 2:
                continue
            d = lev1(a, b)
            if 1 <= d <= 2:
                key = tuple(sorted([a, b]))
                if key not in seen_pairs:
                    seen_pairs.add(key)
                    pairs.append((d, a, new_counts[a], b, new_counts[b]))

    pairs.sort(key=lambda x: (x[0], -(x[2] + x[4])))
    print()
    print(f"SIMILARITY — near-identical pairs (Levenshtein 1-2): {len(pairs)}")
    for d, a, ca, b, cb in pairs:
        print(f"    d={d}  {a!r} ({ca})  ~  {b!r} ({cb})")

    return df_out, change_df, freq_df


df_cleaned, changes, frequencies = run_cleaning(run_pipeline.result, OUTPUT_CSV)